# Batch BH fitting workflow

Run BH fits for an explicit list of `SHOTS` from a data folder.

Preprocessing for fitting is the shared `prepare_bh_fit_arrays` pipeline:

- subtract the mean of `BACKGROUND_FRAMES` for each (frame, channel),
- crop to `BH_FIT_WAVELENGTH_RANGE_NM`,
- compute the normalization scale strictly inside `BH_SCALE_WAVELENGTH_RANGE_NM`,
- divide by that scale (negatives preserved).

The fitter's constant baseline `base` is tightly bounded near zero by default.

`run_folder_batch` resumes by skipping shots whose `summary.csv` already exists in `OUT_DIR/<shot>/`.

In [1]:
from pathlib import Path

from bh_molecule import run_folder_batch

DATA_DIR = Path("~/Dropbox/Experiments/2025-LHD-BH/133mORCA").expanduser()
OUT_DIR = Path("bh_batch_results")

SHOTS = [
    193788,
    193789,
    193790,
]

BACKGROUND_FRAMES = (0, 1, 2, 3)
BH_FIT_WAVELENGTH_RANGE_NM = (433.05, 433.90)
BH_SCALE_WAVELENGTH_RANGE_NM = (433.08, 433.30)

SAVE_FRAMES = False
RUN_FIT_LIMIT = None

CW_NM = 431.91
SCALE = 1.0
TIME_RANGE = (0.0, 10.0)

FRAMES = None
CHANNELS = None

# --- Explicit, reproducible fitter constraints -------------------------------
# Use the calibration notebook (`examples/14_w_inst_calibration.ipynb`) on
# representative spectra to choose `W_INST_DEFAULT` and `W_INST_BOUNDS`.
# Set any of these to `None` to keep the package defaults.
#
# W_INST_DEFAULT  : initial guess (and the fixed value when FIX_W_INST=True)
#                   for the instrumental Gaussian FWHM [nm].
# W_INST_BOUNDS   : tight production bounds [nm]; must satisfy 0 <= lo < hi.
# FIX_W_INST      : if True, w_inst is fixed at W_INST_DEFAULT via parameter
#                   elimination (curve_fit sees only 6 free parameters).
# DX_TOL_NM       : half-width of the allowed wavelength shift dx [nm].
# BASE_BOUND      : half-width of the tight `base` bound applied after
#                   preprocessing.
W_INST_DEFAULT = 0.018692
W_INST_BOUNDS = None
FIX_W_INST = True
DX_TOL_NM = None
BASE_BOUND = None

In [2]:
out_dir = OUT_DIR if OUT_DIR.is_absolute() else (Path.cwd() / OUT_DIR).resolve()

all_fits = sorted(DATA_DIR.glob("*.fits"))
if SHOTS is None:
    selected = all_fits
else:
    wanted = {str(s) for s in SHOTS}
    selected = [p for p in all_fits if p.stem in wanted]
    missing = wanted - {p.stem for p in selected}
    if missing:
        print(f"WARNING: shots {sorted(missing)} not found in {DATA_DIR}")

print(f"Selected data folder:  {DATA_DIR}")
print(f"Output folder:         {out_dir}")
print(f"Selected shots:        {SHOTS}")
print(f"Matched FITS files     ({len(selected)} of {len(all_fits)} total in folder):")
for p in selected:
    print(f"  - {p}")
print(f"Background frames:     {BACKGROUND_FRAMES}")
print(f"BH fit window:         {BH_FIT_WAVELENGTH_RANGE_NM} nm")
print(f"BH scale window:       {BH_SCALE_WAVELENGTH_RANGE_NM} nm")
print(f"SAVE_FRAMES:           {SAVE_FRAMES}")
if SAVE_FRAMES:
    print(
        "  WARNING: SAVE_FRAMES=True is slower and writes per-fit PNGs to "
        f"<out_dir>/<shot>/frames/"
    )
print(f"RUN_FIT_LIMIT:         {RUN_FIT_LIMIT}")
print("--- fitter constraints (None = package default) ---")
print(f"W_INST_DEFAULT:        {W_INST_DEFAULT}")
print(f"W_INST_BOUNDS:         {W_INST_BOUNDS}")
print(f"FIX_W_INST:            {FIX_W_INST}")
print(f"DX_TOL_NM:             {DX_TOL_NM}")
print(f"BASE_BOUND:            {BASE_BOUND}")

Selected data folder:  C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA
Output folder:         C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results
Selected shots:        [193788, 193789, 193790]
Matched FITS files     (3 of 49 total in folder):
  - C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193788.fits
  - C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193789.fits
  - C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193790.fits
Background frames:     (0, 1, 2, 3)
BH fit window:         (433.05, 433.9) nm
BH scale window:       (433.08, 433.3) nm
SAVE_FRAMES:           False
RUN_FIT_LIMIT:         None
--- fitter constraints (None = package default) ---
W_INST_DEFAULT:        0.018692
W_INST_BOUNDS:         None
FIX_W_INST:            True
DX_TOL_NM:             None
BASE_BOUND:            None


In [3]:
results = run_folder_batch(
    DATA_DIR,
    frames=FRAMES,
    channels=CHANNELS,
    shots=SHOTS,
    cw=CW_NM,
    scale=SCALE,
    time_range=TIME_RANGE,
    background_frames=BACKGROUND_FRAMES,
    bh_fit_range=BH_FIT_WAVELENGTH_RANGE_NM,
    bh_scale_range=BH_SCALE_WAVELENGTH_RANGE_NM,
    w_inst_default=W_INST_DEFAULT,
    w_inst_bounds=W_INST_BOUNDS,
    fix_w_inst=FIX_W_INST,
    dx_tol_nm=DX_TOL_NM,
    base_bound=BASE_BOUND,
    out_dir=out_dir,
    save_frames=SAVE_FRAMES,
    run_fit_limit=RUN_FIT_LIMIT,
)
print(f"\nFinished {len(results)} shot(s).")
list(results.keys())

Processing 3 shot(s) under C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA
--- [193788] (1/3) ---
Processing shot 193788: C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193788.fits
Output: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193788
background_frames: (0, 1, 2, 3)
save_frames: False
BH fit window: [433.05, 433.9] nm
BH scale window: [433.08, 433.3] nm
Frame PNG saving disabled.
Background frames look flat (ratio = 0.03)
Detected frames: [6, 7, 8, 9, 10]
Detected channels: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
Using frames [6, 7, 8, 9, 10]
Using channels [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
Fitter constraints: w_inst_default=0.0187 nm, w_inst_bounds=[0.0187, 0.0187] nm, fix_w_inst=True, dx_tol_nm=0.3000,

Fits:   0%|          | 0/200 [00:00<?, ?it/s]

Finished shot 193788
Summary: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193788\summary.csv
Grid: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193788\grid.pdf
--- [193789] (2/3) ---
Processing shot 193789: C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193789.fits
Output: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193789
background_frames: (0, 1, 2, 3)
save_frames: False
BH fit window: [433.05, 433.9] nm
BH scale window: [433.08, 433.3] nm
Frame PNG saving disabled.
Background frames look flat (ratio = 0.03)
Detected frames: [6, 7, 8, 9, 10]
Detected channels: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
Using frames [6, 7, 8, 9, 10]
Using channels [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36

Fits:   0%|          | 0/200 [00:00<?, ?it/s]

Finished shot 193789
Summary: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193789\summary.csv
Grid: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193789\grid.pdf
--- [193790] (3/3) ---
Processing shot 193790: C:\Users\queezz\Dropbox\Experiments\2025-LHD-BH\133mORCA\193790.fits
Output: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193790
background_frames: (0, 1, 2, 3)
save_frames: False
BH fit window: [433.05, 433.9] nm
BH scale window: [433.08, 433.3] nm
Frame PNG saving disabled.
Background frames look flat (ratio = 0.03)
Detected frames: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Detected channels: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]
Using frames [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Using channels [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26

Fits:   0%|          | 0/400 [00:00<?, ?it/s]

Finished shot 193790
Summary: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193790\summary.csv
Grid: C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193790\grid.pdf

Finished 3 shot(s).


['193788', '193789', '193790']

In [4]:
if results:
    first_shot_id = sorted(results.keys())[0]
    print(f"summary.csv for {first_shot_id} -> {out_dir / first_shot_id / 'summary.csv'}")
    display(results[first_shot_id].head())
else:
    print("No new shots were fit (all selected shots already have summary.csv).")

summary.csv for 193788 -> C:\Users\queezz\Dropbox\20-Code\2025-bh-molecule\examples\bh_batch_results\193788\summary.csv


,frame,channel,C,T_rot,dx,w_inst,base,I_R7,I_R8,C_err,T_rot_err,dx_err,w_inst_err,base_err,I_R7_err,I_R8_err,chi2_red,R2,npts
0,6,2,0.306948,8544.782433,0.020941,0.018692,0.03,7.686428e-04,3.046417e-25,0.079700,6441.285023,0.002036,0.0,0.014942,0.001933,0.001933,0.053263,0.110052,356
1,6,3,0.917084,2180.494509,0.020957,0.018692,0.03,3.863743e-03,6.185259e-04,0.175366,474.809932,0.001606,0.0,0.011968,0.001648,0.001648,0.038872,0.233828,356
2,6,4,0.613187,3429.810668,0.018435,0.018692,0.03,3.635941e-15,3.742396e-19,0.071714,609.610961,0.000887,0.0,0.006978,0.000930,0.000930,0.012355,0.561860,356
3,6,5,0.988845,2694.756313,0.016924,0.018692,0.03,2.050794e-03,9.194267e-04,0.086210,305.086029,0.000685,0.0,0.006764,0.000918,0.000918,0.012055,0.705662,356
4,6,6,0.734481,3493.605581,0.016806,0.018692,0.03,1.502822e-03,1.289681e-03,0.086667,634.488083,0.000895,0.0,0.008568,0.001140,0.001140,0.018575,0.422860,356
